# Utility Chains Overview

## Summarizing Documents

### load_summarize_chain

In [1]:
#Please install the package as per your requirement :)
#!pip install openai==1.14.2
#!pip install langchain==0.1.13
#!pip install huggingface-hub==0.21.4
#!pip install langchain-openai==0.1.0
#!pip install tiktoken==0.5.2
#!pip install bs4==0.0.2

In [2]:
import os
os.environ["OPENAI_API_KEY"] = ""

In [3]:
#The below import has been replaced by the later one
#from langchain.llms import OpenAI
from langchain_openai import OpenAI
from langchain.prompts import PromptTemplate
from langchain.chains.summarize import load_summarize_chain
from langchain.text_splitter import CharacterTextSplitter
from langchain.docstore.document import Document

In [4]:
llm = OpenAI(temperature=0.9)

In [6]:
# Reading the document
with open("sample.txt") as f:
    data = f.read()

<font color='green'>
When it comes to document processing, breaking a large document into smaller, more manageable chunks is essential
<font>

In [7]:
# Split text
text_splitter = CharacterTextSplitter()
texts = text_splitter.split_text(data)

In [8]:
# Create multiple documents
docs = [Document(page_content=t) for t in texts]

In [9]:
docs

[Document(page_content="The Evolution and Impact of Computers\n\nComputers have revolutionized the way we live, work, and communicate. From their humble beginnings as massive, room-sized machines to todayâ€™s sleek laptops and powerful smartphones, computers have become an essential part of modern life. Their evolution reflects humanityâ€™s ingenuity and desire to solve complex problems with greater speed and efficiency.\n\nA Brief History\nThe journey of computers began in the mid-20th century with the development of early machines like ENIAC (Electronic Numerical Integrator and Computer), which was used primarily for military calculations. These early computers were bulky, expensive, and limited in function. The invention of the transistor in the late 1940s, followed by the development of integrated circuits in the 1950s and 60s, drastically reduced the size and cost of computers while increasing their power. Personal computers (PCs) emerged in the 1970s and 80s, bringing computing p

<font color='green'>
To create an instance of load_summarizer_chain, we need to provide three arguments. <br><br>Firstly, we need to pass the desired large language model that will be used to query the user input. Secondly, we specify the type of langchain chain to be used for summarizing documents.<br> Lastly, we can set the verbose argument to True if we want to see all the intermediate steps involved in processing the user request and generating the output.<font>

In [10]:
chain = load_summarize_chain(llm, chain_type="map_reduce", verbose=True)
chain.invoke(docs)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"The Evolution and Impact of Computers

Computers have revolutionized the way we live, work, and communicate. From their humble beginnings as massive, room-sized machines to todayâ€™s sleek laptops and powerful smartphones, computers have become an essential part of modern life. Their evolution reflects humanityâ€™s ingenuity and desire to solve complex problems with greater speed and efficiency.

A Brief History
The journey of computers began in the mid-20th century with the development of early machines like ENIAC (Electronic Numerical Integrator and Computer), which was used primarily for military calculations. These early computers were bulky, expensive, and limited in function. The invention of the transistor in the late 1940s, followed by the development of integrated circuits in the 1950s and 60s, drastically reduced the size a

{'input_documents': [Document(page_content="The Evolution and Impact of Computers\n\nComputers have revolutionized the way we live, work, and communicate. From their humble beginnings as massive, room-sized machines to todayâ€™s sleek laptops and powerful smartphones, computers have become an essential part of modern life. Their evolution reflects humanityâ€™s ingenuity and desire to solve complex problems with greater speed and efficiency.\n\nA Brief History\nThe journey of computers began in the mid-20th century with the development of early machines like ENIAC (Electronic Numerical Integrator and Computer), which was used primarily for military calculations. These early computers were bulky, expensive, and limited in function. The invention of the transistor in the late 1940s, followed by the development of integrated circuits in the 1950s and 60s, drastically reduced the size and cost of computers while increasing their power. Personal computers (PCs) emerged in the 1970s and 80s, 

## HTTP Requests

### LLMRequestsChain

In [11]:
from langchain.chains import LLMRequestsChain, LLMChain

In [12]:
template = """
Extract the answer to the question '{query}' or say "not found" if the information is not available.
{requests_result}
"""

PROMPT = PromptTemplate(
    input_variables=["query", "requests_result"],
    template=template,
)

In [13]:
llm=OpenAI()

In [14]:
chain = LLMRequestsChain(llm_chain=LLMChain(llm=llm, prompt=PROMPT))

<font color='green'>
Preparing the question & inputs to the http request<font>

In [15]:
question = "What is the capital of india?"
inputs = {
    "query": question,
    "url": "https://www.google.com/search?q=" + question.replace(" ", "+"),
}

In [16]:
chain.invoke(inputs)

{'query': 'What is the capital of india?',
 'url': 'https://www.google.com/search?q=What+is+the+capital+of+india?',
 'output': '\nNew Delhi'}

<font color='green'>
Let's look at the internal functioning<font>

In [17]:
import inspect
print(inspect.getsource(chain._call))

    def _call(
        self,
        inputs: Dict[str, Any],
        run_manager: Optional[CallbackManagerForChainRun] = None,
    ) -> Dict[str, Any]:
        from bs4 import BeautifulSoup

        _run_manager = run_manager or CallbackManagerForChainRun.get_noop_manager()
        # Other keys are assumed to be needed for LLM prediction
        other_keys = {k: v for k, v in inputs.items() if k != self.input_key}
        url = inputs[self.input_key]
        res = self.requests_wrapper.get(url)
        # extract the text from the html
        soup = BeautifulSoup(res, "html.parser")
        other_keys[self.requests_key] = soup.get_text()[: self.text_length]
        result = self.llm_chain.predict(
            callbacks=_run_manager.get_child(), **other_keys
        )
        return {self.output_key: result}

